### Application of Deep CNN (RESNet), LSTM and Vision Transformer(ViTMAE) for AS detection

In [1]:
# !pip uninstall wandb -y
# !pip install wandb --upgrade

****Import libraries****

In [ ]:
import os
import glob
import shutil
import random
import warnings
from pathlib import Path 
from typing import Tuple, List, Dict, Optional
from collections import Counter

import pandas as pd
import numpy as np
from scipy import signal
from scipy.signal import butter, filtfilt, detrend, decimate, find_peaks
from scipy import stats
from scipy.ndimage import gaussian_filter1d

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import seaborn as sns
import plotly.express as px

import cv2
from skimage.segmentation import mark_boundaries

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from transformers import ViTForImageClassification
from lime import lime_image
import shap
from tqdm import tqdm
import wandb

# Suppress all warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import os
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/local/cuda/'

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Current working directory:", os.getcwd())
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

# using .npy

In [ ]:


class NpyDataset(Dataset):
    def __init__(self, root_dir, transform=None, multiplying_factors=None):
        """
        Args:
            root_dir (string): Directory with all the classes
            transform (callable, optional): Optional transform to be applied
            multiplying_factors (dict): Dictionary mapping class names to multiplication factors
        """
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        
        # Set class names and their index
        self.classes = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.idx_to_class = {idx: cls_name for cls_name, idx in self.class_to_idx.items()}
        
        # Collect all (file_path, label) pairs
        for class_name in self.classes:
            class_dir = self.root_dir / class_name
            for file in class_dir.glob("*.npy"):
                self.samples.append((file, self.class_to_idx[class_name]))
        
        # Apply multiplying factors if provided
        if multiplying_factors:
            self.oversample_classes(multiplying_factors)
        
        self.targets = [label for _, label in self.samples]

    def oversample_classes(self, multiplying_factors):
        """Oversample classes based on multiplying factors"""
        # Group samples by class
        class_samples = {}
        for file_path, label in self.samples:
            class_name = self.idx_to_class[label]
            if class_name not in class_samples:
                class_samples[class_name] = []
            class_samples[class_name].append((file_path, label))
        
        # Create new samples list with oversampling
        new_samples = []
        
        for class_name, samples in class_samples.items():
            # Get multiplication factor for this class (default to 1 if not specified)
            factor = multiplying_factors.get(class_name, 1)
            
            # Add original samples
            new_samples.extend(samples)
            
            # Add oversampled copies
            if factor > 1:
                for _ in range(factor - 1):
                    # Randomly sample from the class with replacement
                    oversampled = random.choices(samples, k=len(samples))
                    new_samples.extend(oversampled)
        
        self.samples = new_samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        array = np.load(file_path)  # shape (1280,)
        
        # Reshape to (1, H, W)
        array = array.reshape(1, 32, 40)  #  (32,40) 
        # Normalize if needed
        array = (array - array.min()) / (array.max() - array.min() + 1e-8)
        
        tensor = torch.tensor(array, dtype=torch.float)
        
        if self.transform:
            tensor = self.transform(tensor)
            
        return tensor, label

    def get_class_distribution(self):
        """Return the distribution of classes in the dataset"""
        return Counter([self.idx_to_class[label] for label in self.targets])

# Load dataset

In [ ]:
# Define multiplying factors based on the paper

multiplying_factors = {
    "AS": 21,    # ×21 for AS  
    "No_AS": 5 # ×5 for no aortic stenosis
}

#Create datasets for train, val, and test
train_dataset = NpyDataset(
    root_dir="/train",
    multiplying_factors=multiplying_factors
)



val_dataset = NpyDataset(
    root_dir="/val", 
    multiplying_factors=multiplying_factors  # Apply same factors to validation
)

test_dataset = NpyDataset(
    root_dir="/test"
    # No multiplying factors for test set - keep original distribution
)

# Check class distribution before and after
print("Original class distribution in train:")
original_counts = {}
for class_name in ["AS", "No_AS"]:
    class_dir = Path("/train") / class_name
    original_counts[class_name] = len(list(class_dir.glob("*.npy")))
print(original_counts)

print("After oversampling distribution:")
print(train_dataset.get_class_distribution())

#Validation set
# Check class distribution before and after
print("Original class distribution in Val:")
original_val_counts = {}
for class_val_name in ["AS", "No_AS"]:
    class_val_dir = Path("val") / class_val_name
    original_val_counts[class_val_name] = len(list(class_val_dir.glob("*.npy")))
print( original_val_counts)

print("After oversampling distribution:")
print(val_dataset.get_class_distribution())

# Create data loaders
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [6]:
#pip install lime shap opencv-python scikit-image

In [7]:
 from lime import lime_tabular

In [8]:
# pip install captum

In [9]:
# !pip install neurokit2

In [10]:
# pip install captum

In [11]:
#pip install captum shap scipy

In [12]:
# pip install captum shap scipy pillow

In [13]:
# !pip install neurokit2


## Resnet18 network

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

# ------------------------------
# ResNet Building Blocks
# ------------------------------
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return F.relu(out)


# ------------------------------
# ResNet-18 Architecture 
# ------------------------------
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=2, in_channels=1, dropout_p=0.3):  #
        super(ResNet, self).__init__()
        self.in_channels = 64

        # First conv layer for 1-channel input
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # Residual layers
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        #  Dropout before FC
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        self._initialize_weights()

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion)
            )
        layers = [block(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)          
        return self.fc(x)


# ------------------------------
# ResNet18 Classifier Wrapper
# ------------------------------
class ResNet18Classifier(nn.Module):
    def __init__(self, num_classes=2, dropout_p=0.3):
        super(ResNet18Classifier, self).__init__()
        self.resnet18 = ResNet(BasicBlock, [2, 2, 2, 2],
                               num_classes=num_classes, in_channels=1, dropout_p=dropout_p)

    def forward(self, x):
        return self.resnet18(x)



In [15]:
# pip install captum


## Train

In [ ]:
# ------------------------------
# Training Code 
# ------------------------------

model = ResNet18Classifier(num_classes=2, dropout_p=0.3).to(device)

num_epochs = 100

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=5, T_mult=2, eta_min=1e-7
)

#  Early stopping settings
patience = 6
best_val_loss = float("inf")
patience_counter = 0
min_delta = 1e-4  # small improvement threshold

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")
    model.train()
    epoch_loss = 0.0
    all_preds, all_labels = [], []

    for batch_idx, (inputs, labels) in enumerate(tqdm(train_loader, desc="Training", leave=False)):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Step scheduler per batch (for cosine warm restarts)
        scheduler.step(epoch + batch_idx / len(train_loader))

        epoch_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_train_loss = epoch_loss / len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    print(f" Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc * 100:.2f}%")

    # ------------------------------
    # Validation phase
    # ------------------------------
    model.eval()
    all_preds, all_labels = [], []
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validating", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    print("\nValidation Results:")
    print(f"Val Loss: {avg_val_loss:.4f}")
    print(classification_report(all_labels, all_preds, target_names=["as", "no_as"]))

    # ------------------------------
    # Early Stopping
    # ------------------------------
    if avg_val_loss < best_val_loss - min_delta:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "resnet18_trained_best.pth")
        print(f" New best model saved! Val Loss improved to {avg_val_loss:.4f}")
    else:
        patience_counter += 1
        print(f" Early stopping counter: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs!")
            break

# Save final model
torch.save(model.state_dict(), "resnet18_trained_final.pth")
print(" Final model saved!")


# Explainability

In [ ]:
# ============================================================================
# CNN EXPLANATIONS - Integrated Gradients (IG) and LIME only
# ============================================================================

import os
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks, butter, filtfilt
from scipy.ndimage import label as nd_label

# ---------------------------
# Unified color scheme for explanation plots
# ---------------------------
# Red-based palette to keep visual interpretation consistent across methods
COLORS = {
    'moderate_impact': '#FFA07A',
    'critical_impact': '#E74C3C',
    'peak_impact': '#8B0000'
}

# ---------------------------
# Optional explanation libraries
# ---------------------------
# Captum is preferred for IG, but a manual fallback is provided
try:
    from captum.attr import IntegratedGradients
    _HAS_CAPTUM = True
except Exception:
    _HAS_CAPTUM = False

# LIME is used for local, perturbation-based explanations
try:
    from lime import lime_tabular
    _HAS_LIME = True
except Exception:
    _HAS_LIME = False


# ---------------------------
# Integrated Gradients (manual fallback)
# ---------------------------
def manual_integrated_gradients_cnn(model, input_tensor, target_class,
                                    device='cpu', steps=50):
    """
    Manual IG implementation used only if Captum is unavailable.
    Keeps behavior consistent with standard IG assumptions.
    """
    model.eval()
    input_tensor = input_tensor.to(device)

    # Zero baseline is standard for normalized ECG inputs
    baseline = torch.zeros_like(input_tensor).to(device)

    alphas = torch.linspace(0.0, 1.0, steps).to(device)
    accumulated_grads = torch.zeros_like(input_tensor).to(device)

    for alpha in alphas:
        interp = baseline + alpha * (input_tensor - baseline)
        interp.requires_grad_(True)

        output = model(interp)
        score = output[:, target_class].sum()

        grads = torch.autograd.grad(
            outputs=score,
            inputs=interp,
            retain_graph=False,
            create_graph=False
        )[0]

        accumulated_grads += grads

    avg_grads = accumulated_grads / steps
    attributions = (input_tensor - baseline) * avg_grads
    return attributions.detach().cpu()


# ---------------------------
# ECG feature detection (for visualization context)
# ---------------------------
def enhanced_ecg_feature_detection(ecg_signal, fs=256):
    """
    Lightweight ECG feature detection to provide physiological context
    in explanation plots (not used by the model itself).
    """
    try:
        # Band-pass filter focused on QRS energy
        b, a = butter(4, [5, 15], btype='band', fs=fs)
        filtered_ecg = filtfilt(b, a, ecg_signal)
    except Exception:
        filtered_ecg = ecg_signal

    features = {
        'qrs_peaks': [],
        'p_waves': [],
        't_waves': [],
        'st_segments': [],
        'st_t_waves': [],
        'qt_intervals': []
    }

    try:
        rms = np.sqrt(np.mean(filtered_ecg ** 2))
        threshold = rms * 2.5

        r_peaks, _ = find_peaks(
            filtered_ecg,
            height=threshold,
            distance=fs // 3,
            prominence=rms * 1.5
        )

        # Simple sanity check to discard noisy detections
        valid_r_peaks = []
        for peak in r_peaks:
            if 50 < peak < len(ecg_signal) - 50:
                segment = ecg_signal[peak - 25: peak + 25]
                if np.ptp(segment) > rms:
                    valid_r_peaks.append(peak)

        features['qrs_peaks'] = valid_r_peaks

    except Exception:
        features['qrs_peaks'], _ = find_peaks(ecg_signal, distance=fs // 3)

    # Approximate timing windows around each R-peak
    for r_peak in features['qrs_peaks']:
        p_start = max(0, r_peak - int(0.25 * fs))
        p_end = max(0, r_peak - int(0.12 * fs))
        if p_end > p_start:
            features['p_waves'].append((p_start, p_end))

        qrs_start = max(0, r_peak - int(0.06 * fs))
        qrs_end = min(len(ecg_signal) - 1, r_peak + int(0.06 * fs))

        st_start = qrs_end
        st_end = min(len(ecg_signal) - 1, r_peak + int(0.12 * fs))
        if st_end > st_start:
            features['st_segments'].append((st_start, st_end))

        t_start = min(len(ecg_signal) - 1, r_peak + int(0.12 * fs))
        t_end = min(len(ecg_signal) - 1, r_peak + int(0.4 * fs))
        if t_end > t_start:
            features['t_waves'].append((t_start, t_end))
            features['st_t_waves'].append((st_start, t_end))
            features['qt_intervals'].append((qrs_start, t_end))

    return features


# ---------------------------
# ECG preprocessing for CNN input
# ---------------------------
def process_ecg_for_cnn(ecg_signal, target_shape=(32, 40)):
    """
    Converts a 1D ECG signal into a fixed-size 2D representation
    compatible with CNN-based models.
    """
    if ecg_signal.ndim > 1:
        ecg_signal = ecg_signal.flatten()

    # Min–max normalization for numerical stability
    ecg_min, ecg_max = ecg_signal.min(), ecg_signal.max()
    ecg_norm = (ecg_signal - ecg_min) / (ecg_max - ecg_min + 1e-8)

    target_size = target_shape[0] * target_shape[1]

    # Pad or truncate to match CNN input size
    ecg_padded = np.pad(
        ecg_norm,
        (0, max(0, target_size - len(ecg_norm))),
        mode='edge'
    )[:target_size]

    ecg_2d = ecg_padded.reshape(target_shape)

    # Single-channel input (C, H, W)
    return torch.tensor(ecg_2d, dtype=torch.float32).unsqueeze(0)


def process_cnn_attributions(attributions, original_signal_len):
    """
    Maps 2D CNN attributions back to a 1D importance signal
    aligned with the original ECG.
    """
    attr_2d = attributions[0].mean(0).flatten().numpy()
    attr_abs = np.abs(attr_2d)

    # Normalize for visualization
    attr_norm = (attr_abs - attr_abs.min()) / (attr_abs.max() - attr_abs.min() + 1e-12)

    return np.interp(
        np.linspace(0, 1, original_signal_len),
        np.linspace(0, 1, len(attr_norm)),
        attr_norm
    )


def get_model_output(model, input_tensor):
    """Wrapper to keep output handling consistent."""
    return model(input_tensor)


# ---------------------------
# LIME implementation for CNN-based ECG models
# ---------------------------
def create_lime_explanation_simple(model, device, ecg_signal, pred_class,
                                   num_samples=500, num_segments=50):
    """
    Segment-based LIME explanation where contiguous ECG regions
    are masked and perturbed.
    """
    orig_len = len(ecg_signal)
    segment_size = orig_len // num_segments

    segments = [
        (i * segment_size, min(orig_len, (i + 1) * segment_size))
        for i in range(num_segments)
    ]

    def predict_fn(X):
        predictions = []
        for mask in X:
            perturbed = ecg_signal.copy()

            # Replace masked segments with local mean
            for j, (start, end) in enumerate(segments):
                if mask[j] == 0:
                    perturbed[start:end] = np.mean(ecg_signal[start:end])

            tensor = process_ecg_for_cnn(perturbed).unsqueeze(0).to(device)

            with torch.no_grad():
                probs = F.softmax(model(tensor), dim=1).cpu().numpy()

            predictions.append(probs[0])

        return np.array(predictions)

    # Random binary masks used as LIME background
    background = np.random.choice(
        [0, 1],
        size=(300, num_segments),
        p=[0.4, 0.6]
    )

    explainer = lime_tabular.LimeTabularExplainer(
        training_data=background,
        mode='classification',
        feature_names=[f"seg_{i}" for i in range(num_segments)],
        discretize_continuous=True,
        random_state=42
    )

    exp = explainer.explain_instance(
        np.ones(num_segments),
        predict_fn,
        num_features=num_segments,
        num_samples=num_samples,
        labels=(pred_class,)
    )

    return exp, segments


# ---------------------------
# Main explanation pipeline
# ---------------------------
def explain_single_file_improved(
    model,
    device,
    file_path,
    class_names,
    fs=256,
    save_dir='/final_explanations',
    ig_steps=50,
    lime_num_samples=500
):
    """
    Generates IG and LIME explanations for a single ECG sample.
    """
    os.makedirs(save_dir, exist_ok=True)

    ecg_signal = np.load(file_path).flatten()
    tensor = process_ecg_for_cnn(ecg_signal).unsqueeze(0).to(device)

    # Model prediction
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        pred_class = int(output.argmax(1))
        confidence = float(probs[0, pred_class])

    pred_name = class_names[pred_class]
    time_axis = np.linspace(0, len(ecg_signal) / fs, len(ecg_signal))

    # ECG feature detection (for interpretability context)
    features = enhanced_ecg_feature_detection(ecg_signal, fs)

    # Integrated Gradients
    if _HAS_CAPTUM:
        ig = IntegratedGradients(model)
        attributions = ig.attribute(tensor, target=pred_class, n_steps=ig_steps)
    else:
        attributions = manual_integrated_gradients_cnn(
            model, tensor, pred_class, device, ig_steps
        )

    importance = process_cnn_attributions(attributions, len(ecg_signal))

    # LIME explanation
    exp, segments = create_lime_explanation_simple(
        model,
        device,
        ecg_signal,
        pred_class,
        num_samples=lime_num_samples
    )

    return {
        'prediction': pred_name,
        'confidence': confidence,
        'beats_detected': len(features['qrs_peaks'])
    }


In [ ]:

model.eval()
results = explain_single_file_improved(
    model=model,
    device=device,
    file_path="filepath",
    class_names=train_dataset.classes,
    fs=256,
    save_dir='/final_explanations',
    ig_steps=50,
    lime_num_samples=1000,
    
)
print(results)

## Evaluate the model

In [12]:


import torch.nn.functional as F

y_test = []
y_test_pred = []
y_test_probs = []  # store softmax probabilities for analysis

for i, data in enumerate(test_loader, 0):
    inputs, y_test_temp = data
    inputs = inputs.to(device)
    with torch.no_grad():
        outputs = model(inputs)
        probs = F.softmax(outputs, dim=1)  # convert logits to probabilities
        preds = torch.argmax(probs, dim=1)

    y_test.extend(y_test_temp.cpu().numpy())
    y_test_pred.extend(preds.cpu().numpy())
    y_test_probs.extend(probs[:, 1].cpu().numpy())  # prob of AS class (if AS=1)


## Evaluation Metrics

In [ ]:



classes = test_loader.dataset.classes #  actual class names

# y_test and y_test_pred 
accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)

print(f'✅ Accuracy:  {accuracy * 100:.2f}%')
print(f'✅ Precision: {precision * 100:.2f}%')
print(f'✅ Recall:    {recall * 100:.2f}%')
print(f'✅ F1 Score:  {f1 * 100:.2f}%')

# Also calculate per-class metrics for better insight
precision_per_class = precision_score(y_test, y_test_pred, average=None)
recall_per_class = recall_score(y_test, y_test_pred, average=None)
f1_per_class = f1_score(y_test, y_test_pred, average=None)

print(f"\n Per-class Metrics:")
for i, class_name in enumerate(classes):
    print(f"   {class_name}: Precision: {precision_per_class[i]:.3f}, Recall: {recall_per_class[i]:.3f}, F1: {f1_per_class[i]:.3f}")

# Classification report
print("\n📊 Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=classes))

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Additional: Calculate balanced accuracy for imbalanced datasets
from sklearn.metrics import balanced_accuracy_score
balanced_acc = balanced_accuracy_score(y_test, y_test_pred)
print(f"✅ Balanced Accuracy: {balanced_acc * 100:.2f}%")

# Custom Class for LSTM 

In [ ]:
import os
import glob
import shutil
import random
import warnings
from pathlib import Path 
from typing import Tuple, List, Dict, Optional
from collections import Counter

import pandas as pd
import numpy as np
from scipy import signal
from scipy.signal import butter, filtfilt, detrend, decimate, find_peaks
from scipy import stats
from scipy.ndimage import gaussian_filter1d

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import seaborn as sns
import plotly.express as px

import cv2
from skimage.segmentation import mark_boundaries

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from transformers import ViTForImageClassification
from lime import lime_image
import shap
from tqdm import tqdm
import wandb

# Suppress all warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import os
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/local/cuda/'

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Current working directory:", os.getcwd())
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path
import random
from collections import Counter
import math

# ----------------------------------------------------------
# Global stats computation (Welford algorithm)
# ----------------------------------------------------------
def compute_global_stats(root_dir, target_len=1280):
    """
    Compute mean and std across all files using Welford's algorithm.
    Pads/crops signals to target_len before aggregation.
    """
    root_dir = Path(root_dir)
    files = list(root_dir.rglob("*.npy"))
    if not files:
        raise ValueError("No .npy files found in the dataset path")

    mean, m2, n_total = 0.0, 0.0, 0

    for f in files:
        arr = np.load(f).reshape(-1, 1).astype(np.float64)
        T = arr.shape[0]
        if T < target_len:
            arr = np.pad(arr, ((0, target_len - T), (0, 0)), mode="constant")
        elif T > target_len:
            arr = arr[:target_len]

        flat = arr.ravel()
        for x in flat:
            n_total += 1
            delta = x - mean
            mean += delta / n_total
            delta2 = x - mean
            m2 += delta * delta2

    std = math.sqrt(m2 / (n_total - 1)) if n_total > 1 else 1.0
    print(f"[Global Stats] mean={mean:.8f}, std={std:.8f} from {len(files)} files ({n_total} samples)")
    return mean, std

# ----------------------------------------------------------
# Dataset class supporting CNN / ViT / LSTM
# ----------------------------------------------------------
class NpyDataset(Dataset):
    def __init__(self,
                 root_dir,
                 mode="cnn",             # "cnn", "vit", or "lstm"
                 target_len=1280,
                 multiplying_factors=None,  # e.g., {"AS":21, "No_AS":5}
                 augment=True,
                 global_stats=None,
                 transform=None):
        self.root_dir = Path(root_dir)
        self.mode = mode
        self.target_len = target_len
        self.augment_flag = augment
        self.global_stats = global_stats
        self.transform = transform

        # ---- Class discovery ----
        self.classes = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}

        # ---- Gather original samples ----
        self.samples = []
        for cls in self.classes:
            for f in (self.root_dir / cls).glob("*.npy"):
                self.samples.append((f, self.class_to_idx[cls], random.randint(0, 2**31-1)))

        # ---- Oversampling ----
        if multiplying_factors:
            self._oversample(multiplying_factors)

        self.targets = [lbl for _, lbl, _ in self.samples]
        print(f"[Init] Dataset length after oversampling: {len(self.samples)}")

    # ------------------------------------------------------
    def _oversample(self, factors):
        """
        Real oversampling:
        - Preserves originals
        - Adds duplicates according to multiplying_factors
        - Each duplicate gets a unique seed
        """
        per_class = {}
        for path, lbl, seed in self.samples:
            cls_name = self.idx_to_class[lbl]
            per_class.setdefault(cls_name, []).append((path, lbl))

        new_samples = []
        for cls in self.classes:
            files = per_class.get(cls, [])
            factor = factors.get(cls, 1)
            # always keep originals
            new_samples.extend([(p, lbl, random.randint(0, 2**31-1)) for p, lbl in files])
            if factor > 1:
                for _ in range(factor-1):
                    # randomly sample duplicates with replacement
                    for p, lbl in random.choices(files, k=len(files)):
                        new_samples.append((p, lbl, random.randint(0, 2**31-1)))
        self.samples = new_samples

    # ------------------------------------------------------
    def _pad_or_crop(self, sig):
        T = sig.shape[0]
        if T < self.target_len:
            sig = np.pad(sig, ((0, self.target_len - T), (0,0)), mode="constant")
        elif T > self.target_len:
            sig = sig[:self.target_len]
        return sig

    # ------------------------------------------------------
    def _augment_signal(self, sig, rng):
        L = sig.shape[0]

        # Amplitude scaling
        if rng.rand() < 0.3:
            sig *= rng.uniform(0.95, 1.05)
    
        # Gaussian noise
        if rng.rand() < 0.3:
            sig += 0.003 * rng.randn(*sig.shape)
    
        # Small shift (removed for now because it might be harmful)
        # if rng.rand() < 0.2:
        #    shift = int(rng.uniform(-0.01, 0.01) * L)
        #    sig = np.roll(sig, shift, axis=0)
    
        return sig

        # # Tiny sinusoidal drift
        # if rng.rand() < 0.2:
        #     mag = rng.uniform(0, 0.02)
        #     freq = rng.uniform(0.001, 0.008)
        #     t = np.arange(L).reshape(-1, 1)
        #     sig += mag * np.sin(2 * np.pi * freq * t)



    # ------------------------------------------------------
    def __getitem__(self, idx):
        path, label, seed = self.samples[idx]
        arr = np.load(path).reshape(-1, 1).astype(np.float32)
        arr = self._pad_or_crop(arr)

        rng = np.random.RandomState(seed + idx)

        # ---- Augmentation only on some samples

        # label is integer
        label_name = self.idx_to_class[label]
        if self.augment_flag:
            # default 0.4, but for heavily oversampled minority you might use 0.5
            default_aug_prob = 0.5
            # give minority slightly higher chance but not too high
            if label_name == "AS":
                aug_prob = 0.5
            else:
                aug_prob = 0.35
            if rng.rand() < aug_prob:
                arr = self._augment_signal(arr, rng)
                arr = self._pad_or_crop(arr)


        # ---- Normalization
        if self.global_stats is not None:
            mean, std = self.global_stats
            std = max(std, 1e-6)
            arr = (arr - mean) / std
        else:
            mean, std = arr.mean(), arr.std()
            std = max(std, 1e-6)
            arr = (arr - mean) / std
        # if self.global_stats is None:
        #     # mean, std = arr.mean(), arr.std()
        #     # std = max(std, 1e-6)
        #     # arr = (arr - mean) / std
        #     pass

        # ---- Reshape for model
        if self.mode == "cnn":
            arr = arr.reshape(1, 32, 40)
        elif self.mode == "vit":
            arr = arr.reshape(1, 32, 40)
            arr = np.repeat(arr, 3, axis=0)
        elif self.mode == "lstm":
            # For LSTM: (seq_len, input_dim=1)
            arr = arr.reshape(self.target_len, 1)
        else:
            raise ValueError(f"Unknown mode: {self.mode}")

        tensor = torch.tensor(arr, dtype=torch.float32)
        if self.transform:
            tensor = self.transform(tensor)
        return tensor, label

    # ------------------------------------------------------
    def __len__(self):
        return len(self.samples)

    def get_class_distribution(self):
        return Counter([self.idx_to_class[l] for l in self.targets])


# Load Dataset

In [ ]:



# ============================================================================
# DATASET SETUP WITH NORMALIZATION 
# ============================================================================
train_root = "/train"
val_root   = "/val"
test_root  = "/test"

# Compute global stats (MUST DO THIS!)
print("Computing global statistics...")
mean, std = compute_global_stats(train_root, target_len=1280)
print(f"✓ Global stats: mean={mean:.6f}, std={std:.6f}")

# Oversampling factors
multiplying_factors = {"AS": 21, "No_AS": 5}

# Create datasets WITH normalization
mode = "lstm"
train_dataset = NpyDataset(
    root_dir=train_root,
    mode=mode,
    target_len=1280,
    multiplying_factors=multiplying_factors,
    augment=True,
    global_stats=(mean, std)  # ENABLE NORMALIZATION!
)

val_dataset = NpyDataset(
    root_dir=val_root,
    mode=mode,
    target_len=1280,
    augment=False,
    global_stats=(mean, std)  #  ENABLE NORMALIZATION!
)

test_dataset = NpyDataset(
    root_dir=test_root,
    mode=mode,
    target_len=1280,
    augment=False,
    global_stats=(mean, std)  #  ENABLE NORMALIZATION!
)

print("Train class distribution:", train_dataset.get_class_distribution())

# Create DataLoaders
batch_size = 32  # Increased from 16 for better gradient estimates
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Check data shape
for x, y in train_loader:
    print(f"✓ Data shape: {x.shape}, Labels: {y.shape}")
    break






# LSTM 

In [ ]:


# ============================================================================
#  MEMORY-FRIENDLY LSTM MODEL
# ============================================================================
class ECG_LSTM_Light(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2, num_classes=2, dropout=0.3):
        super(ECG_LSTM_Light, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )

        self.fc = nn.Sequential(
            nn.BatchNorm1d(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(dropout / 2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        elif x.dim() == 4:
            x = x.view(x.size(0), -1, 1)

        lstm_out, (hn, cn) = self.lstm(x)
        last_hidden = hn[-1]
        output = self.fc(last_hidden)
        return output


# ============================================================================
# TRAINING SETUP (Mixed Precision + Early Stopping)
# ============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model
model = ECG_LSTM_Light(
    input_size=1,
    hidden_size=128,
    num_layers=2,
    num_classes=2,
    dropout=0.3
).to(device)

# Loss, optimizer, and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)

# Mixed precision setup
scaler = torch.cuda.amp.GradScaler()

# Early stopping params
num_epochs = 50
patience = 7
best_val_f1 = 0.0
patience_counter = 0

print(f"\n{'='*70}")
print(f"Starting Mixed Precision Training for {num_epochs} epochs")
print(f"{'='*70}\n")


# ============================================================================
# TRAINING LOOP (with torch.cuda.amp)
# ============================================================================
for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch [{epoch + 1}/{num_epochs}]")
    print(f"{'='*50}")

    # ===== TRAINING PHASE =====
    model.train()
    epoch_loss = 0.0
    all_preds, all_labels = [], []

    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():  # ✅ Mixed precision
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = epoch_loss / len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    train_f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)

    # ===== VALIDATION PHASE =====
    model.eval()
    val_loss, val_preds, val_labels = 0.0, [], []

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validation", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            with torch.cuda.amp.autocast():  # ✅ Mixed precision
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(val_labels, val_preds)
    val_precision = precision_score(val_labels, val_preds, average='binary', zero_division=0)
    val_recall = recall_score(val_labels, val_preds, average='binary', zero_division=0)
    val_f1 = f1_score(val_labels, val_preds, average='binary', zero_division=0)

    print(f" Train Loss: {avg_loss:.4f} | Train Acc: {train_acc*100:.2f}% | F1: {train_f1:.4f}")
    print(f" Val Loss:   {avg_val_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Val F1: {val_f1:.4f}")

    scheduler.step()
    print(f" LR: {optimizer.param_groups[0]['lr']:.2e}")

    # ===== EARLY STOPPING =====
    if val_f1 > best_val_f1 + 0.005:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_lstm_light.pth")
        print(f" New best model saved! (Val F1: {val_f1:.4f})")
    else:
        patience_counter += 1
        print(f" Early stopping counter: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"\n Early stopping triggered after {epoch+1} epochs!")
        break


# ============================================================================
# LOAD BEST MODEL + TEST EVALUATION
# ============================================================================
model.load_state_dict(torch.load("best_lstm_light.pth"))
model.eval()
print("\n Best model loaded for testing.")

test_preds, test_labels = [], []
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        with torch.cuda.amp.autocast():
            outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_labels, test_preds)
test_precision = precision_score(test_labels, test_preds, average='binary', zero_division=0)
test_recall = recall_score(test_labels, test_preds, average='binary', zero_division=0)
test_f1 = f1_score(test_labels, test_preds, average='binary', zero_division=0)

print(f"\n FINAL TEST RESULTS:")
print(f"Accuracy:  {test_acc*100:.2f}%")
print(f"Precision: {test_precision*100:.4f}")
print(f"Recall:    {test_recall*100:.4f}")
print(f"F1 Score:  {test_f1*100:.4f}")


# Explainability

In [ ]:
# ============================================================================
# LSTM ECG EXPLANATIONS
# Methods: Integrated Gradients (IG) + LIME
# ============================================================================

import os
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.signal import find_peaks, butter, filtfilt
from scipy.ndimage import gaussian_filter1d
from scipy.ndimage import label as nd_label

# ============================================================================
# Optional explanation libraries
# ============================================================================

try:
    from captum.attr import IntegratedGradients
    _HAS_CAPTUM = True
except Exception:
    _HAS_CAPTUM = False

try:
    from lime import lime_tabular
    _HAS_LIME = True
except Exception:
    _HAS_LIME = False


# ============================================================================
# Visualization colors
# ============================================================================

COLORS = {
    "moderate": "#FFA07A",
    "high": "#E74C3C",
    "peak": "#8B0000"
}


# ============================================================================
# Integrated Gradients (manual fallback for LSTM)
# ============================================================================

def manual_integrated_gradients_lstm(
    model,
    input_tensor,
    target_class,
    device="cpu",
    steps=50
):
    """
    Manual Integrated Gradients implementation for LSTM models.

    This version avoids cuDNN backward errors and BatchNorm issues
    when explaining a single ECG sample.
    """
    was_training = model.training

    # LSTM backward pass requires train mode
    model.train()

    # BatchNorm layers fail with batch_size=1 in train mode
    for m in model.modules():
        if isinstance(
            m,
            (
                torch.nn.BatchNorm1d,
                torch.nn.BatchNorm2d,
                torch.nn.BatchNorm3d,
            ),
        ):
            m.eval()

    input_tensor = input_tensor.to(device)
    baseline = torch.zeros_like(input_tensor)

    alphas = torch.linspace(0, 1, steps).to(device)
    total_grads = torch.zeros_like(input_tensor)

    for alpha in alphas:
        interpolated = baseline + alpha * (input_tensor - baseline)
        interpolated.requires_grad_(True)

        output = model(interpolated)
        score = output[:, target_class].sum()

        grads = torch.autograd.grad(
            score,
            interpolated,
            retain_graph=False,
            create_graph=False,
        )[0]

        total_grads += grads

    avg_grads = total_grads / steps
    attributions = (input_tensor - baseline) * avg_grads

    if not was_training:
        model.eval()

    return attributions.detach().cpu()


# ============================================================================
# ECG Feature Detection (rule-based)
# ============================================================================

def enhanced_ecg_feature_detection(ecg_signal, fs=256):
    """
    Detect ECG features using timing rules relative to R-peaks.
    This is used for visualization and clinical context only.
    """
    try:
        b, a = butter(4, [5, 15], btype="band", fs=fs)
        filtered = filtfilt(b, a, ecg_signal)
    except Exception:
        filtered = ecg_signal

    features = {
        "qrs_peaks": [],
        "p_waves": [],
        "t_waves": [],
        "st_segments": [],
        "st_t_waves": [],
        "qt_intervals": [],
    }

    rms = np.sqrt(np.mean(filtered ** 2))
    threshold = rms * 2.5

    r_peaks, _ = find_peaks(
        filtered,
        height=threshold,
        distance=fs // 3,
        prominence=rms * 1.5,
    )

    valid_peaks = []
    for r in r_peaks:
        if 50 < r < len(ecg_signal) - 50:
            seg = ecg_signal[r - 25 : r + 25]
            if np.ptp(seg) > rms:
                valid_peaks.append(r)

    features["qrs_peaks"] = valid_peaks

    for r in valid_peaks:
        p_start, p_end = r - int(0.25 * fs), r - int(0.12 * fs)
        qrs_start, qrs_end = r - int(0.06 * fs), r + int(0.06 * fs)
        st_start, st_end = qrs_end, r + int(0.12 * fs)
        t_start, t_end = r + int(0.12 * fs), r + int(0.40 * fs)

        features["p_waves"].append((max(0, p_start), max(0, p_end)))
        features["st_segments"].append((st_start, st_end))
        features["t_waves"].append((t_start, t_end))
        features["st_t_waves"].append((st_start, t_end))
        features["qt_intervals"].append((qrs_start, t_end))

    return features


# ============================================================================
# ECG preprocessing for LSTM
# ============================================================================

def process_ecg_for_lstm(ecg_signal, target_length=1280, global_stats=None):
    """
    Pad/crop ECG and normalize using training statistics.
    """
    ecg_signal = ecg_signal.flatten()

    if len(ecg_signal) < target_length:
        ecg_signal = np.pad(
            ecg_signal,
            (0, target_length - len(ecg_signal)),
            mode="edge",
        )
    else:
        ecg_signal = ecg_signal[:target_length]

    if global_stats is not None:
        mean, std = global_stats
        ecg_signal = (ecg_signal - mean) / (std + 1e-8)
    else:
        ecg_signal = (ecg_signal - ecg_signal.min()) / (
            ecg_signal.ptp() + 1e-8
        )

    return (
        torch.tensor(ecg_signal, dtype=torch.float32)
        .unsqueeze(0)
        .unsqueeze(-1)
    )


def process_lstm_attributions(attributions, original_len):
    """
    Convert LSTM attributions to a 1D importance signal.
    """
    attr = attributions[0].mean(-1).abs().numpy()

    if attr.max() > attr.min():
        attr = (attr - attr.min()) / (attr.max() - attr.min())

    return np.interp(
        np.linspace(0, 1, original_len),
        np.linspace(0, 1, len(attr)),
        attr,
    )


# ============================================================================
# LIME for ECG
# ============================================================================

def create_lime_explanation(
    model,
    device,
    ecg_signal,
    pred_class,
    num_samples=500,
    num_segments=40,
    global_stats=None,
    target_length=1280,
):
    """
    Segment-based LIME explanation for 1D ECG signals.
    """
    model.eval()
    length = len(ecg_signal)
    seg_size = length // num_segments

    segments = [
        (
            i * seg_size,
            length if i == num_segments - 1 else (i + 1) * seg_size,
        )
        for i in range(num_segments)
    ]

    def predict_fn(X):
        preds = []
        for mask in X:
            signal = ecg_signal.copy()
            for i, (s, e) in enumerate(segments):
                if mask[i] == 0:
                    signal[s:e] = np.mean(ecg_signal[s:e])

            tensor = process_ecg_for_lstm(
                signal, target_length, global_stats
            ).to(device)

            with torch.no_grad():
                prob = F.softmax(model(tensor), dim=1).cpu().numpy()

            preds.append(prob[0])
        return np.array(preds)

    background = np.random.choice(
        [0, 1], size=(300, num_segments), p=[0.4, 0.6]
    )

    explainer = lime_tabular.LimeTabularExplainer(
        training_data=background,
        mode="classification",
        feature_names=[f"seg_{i}" for i in range(num_segments)],
        random_state=42,
    )

    explanation = explainer.explain_instance(
        np.ones(num_segments),
        predict_fn,
        num_features=num_segments,
        num_samples=num_samples,
        labels=(pred_class,),
    )

    return explanation, segments


# ============================================================================
# Global statistics (must match training preprocessing)
# ============================================================================

def compute_global_stats(root_dir, target_len=1280):
    """
    Compute dataset-wide mean and std for ECG normalization.
    """
    root_dir = Path(root_dir)
    files = list(root_dir.rglob("*.npy"))

    if not files:
        raise RuntimeError("No ECG files found.")

    mean, m2, n = 0.0, 0.0, 0

    for f in files:
        arr = np.load(f).reshape(-1)

        if len(arr) < target_len:
            arr = np.pad(arr, (0, target_len - len(arr)), mode="constant")
        else:
            arr = arr[:target_len]

        for x in arr:
            n += 1
            delta = x - mean
            mean += delta / n
            m2 += delta * (x - mean)

    std = math.sqrt(m2 / (n - 1))
    print(f"[Global stats] mean={mean:.6f}, std={std:.6f}")

    return mean, std


# ============================================================================
# Main explanation pipeline
# ============================================================================

def explain_single_file(
    model,
    device,
    file_path,
    class_names,
    global_stats,
    target_len=1280,
    fs=256,
    ig_steps=50,
    lime_num_samples=500,
):
    """
    Generate IG and LIME explanations for a single ECG file.
    """
    ecg = np.load(file_path).flatten()

    input_tensor = process_ecg_for_lstm(
        ecg, target_len, global_stats
    ).to(device)

    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probs = F.softmax(output, dim=1)
        pred_class = int(output.argmax(1))
        confidence = float(probs[0, pred_class])

    pred_name = class_names[pred_class]

    # Integrated Gradients
    if _HAS_CAPTUM:
        ig = IntegratedGradients(model)
        attr = ig.attribute(
            input_tensor, target=pred_class, n_steps=ig_steps
        )
    else:
        attr = manual_integrated_gradients_lstm(
            model, input_tensor, pred_class, device, ig_steps
        )

    importance = process_lstm_attributions(attr, len(ecg))

    # LIME
    lime_exp, lime_segments = create_lime_explanation(
        model,
        device,
        ecg,
        pred_class,
        lime_num_samples,
        40,
        global_stats,
        target_len,
    )

    features = enhanced_ecg_feature_detection(ecg, fs)

    return {
        "prediction": pred_name,
        "confidence": confidence,
        "beats_detected": len(features["qrs_peaks"]),
        "features": features,
        "ig_importance": importance,
        "lime_explanation": lime_exp,
    }


# ============================================================================
# USAGE EXAMPLE (REPRODUCIBLE)
# ============================================================================

#  Compute global stats (must match training)
mean, std = compute_global_stats(
    "/train",
    target_len=1280,
)

#  Run explanation
model.eval()

results = explain_single_file(
    model=model,
    device=device,
    file_path="/filepath",
    class_names=train_dataset.classes,
    global_stats=(mean, std),
    target_len=1280,
    fs=256,
    ig_steps=50,
    lime_num_samples=500,
)

print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(f"Prediction: {results['prediction']}")
print(f"Confidence: {results['confidence']:.2%}")
print(f"Beats detected: {results['beats_detected']}")


# Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)


# === Final Test Evaluation ===
model.eval()
y_test, y_test_pred, y_test_prob = [], [], []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs = inputs.to(device)
        outputs = model(inputs)                 # logits
        probs = torch.softmax(outputs, dim=1)   # class probabilities

        y_test.extend(labels.cpu().numpy())
        y_test_pred.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        y_test_prob.extend(probs[:, 1].cpu().numpy())  # probability of positive (AS) class

y_test       = np.array(y_test)
y_test_pred  = np.array(y_test_pred)
y_test_prob  = np.array(y_test_prob)

# === Basic Metrics ===
accuracy      = accuracy_score(y_test, y_test_pred)
balanced_acc  = balanced_accuracy_score(y_test, y_test_pred)
precision_mac = precision_score(y_test, y_test_pred)
recall_mac    = recall_score(y_test, y_test_pred)
f1_mac        = f1_score(y_test, y_test_pred)

# === ROC-AUC & PR-AUC ===
# If you have >2 classes you'll need a one-vs-rest strategy.
roc_auc = roc_auc_score(y_test, y_test_prob)
pr_auc  = average_precision_score(y_test, y_test_prob)

print("\n Final Test Results")
print(f"✅ Accuracy:           {accuracy * 100:.2f}%")
print(f"✅ Balanced Accuracy:  {balanced_acc * 100:.2f}%")
print(f"✅ Precision (macro):  {precision_mac * 100:.2f}%")
print(f"✅ Recall (macro):     {recall_mac * 100:.2f}%")
print(f"✅ F1 Score (macro):   {f1_mac * 100:.2f}%")
print(f"✅ ROC-AUC:            {roc_auc * 100:.2f}%")
print(f"✅ PR-AUC:             {pr_auc * 100:.2f}%")

# === Classification report ===
# Grab class names from dataset to avoid NameError
classes = test_loader.dataset.classes
print("\n Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=classes))

# === Confusion Matrix ===
conf_matrix = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()
